In [1]:
import json

In [ ]:
!pip install gdown

In [2]:


with open('wholebody.json','r') as whole_body_json:
    content = json.load(whole_body_json)
whole_body_imgs = content['images']
whole_body = content['annotations']


In [3]:
import os
len(os.listdir('jsons/jsons-23-46'))

149813

In [4]:
def find_max(arr):
    mx = -1
    mi = -1
    for a in range(len(arr)):
        if arr[a][2] > mx:
            mx = arr[a][2]
            mi = a
    return mi

In [5]:
req_kps = ['num_keypoints','keypoints','image_id','category_id','id',]

In [6]:
from tqdm.notebook import tqdm




files = os.listdir('jsons/jsons-23-46')
new_annotations = []
for n,f in tqdm(enumerate(files),total=len(files)):
    with open(os.path.join('jsons/jsons-23-46',f)) as file:
        content = json.load(file)
        keypoints = content['key_points']
        del keypoints['left_eye']
        del keypoints['right_eye']
        del keypoints['left_ear']
        del keypoints['right_ear']
#         del keypoints['nose']

        keypoints = list(keypoints.values())
        n_kps = 0
        for k in keypoints:
            if k[2] !=0:
                n_kps += 1
        img_id = content['image_id']
        for an in whole_body:
            if an['image_id'] == int(img_id):
                lkp = [0,0,0]
                rkp = [0,0,0]
                if an['lefthand_valid']:
                    kk = an['lefthand_kpts'][9:]
                    karr = [kk[i:i+3] for i in range(0,len(kk),3)]
                    mi = find_max(karr)
                    lkp = karr[mi]
                    lkp[2] = 2 if lkp[2] > .5 else 1
                    n_kps += 1
                if an['righthand_valid']:
                    kk = an['righthand_kpts'][9:]
                    karr = [kk[i:i+3] for i in range(0,len(kk),3)]
                    mi = find_max(karr)
                    rkp = karr[mi]
                    rkp[2] = 2 if rkp[2] > .5 else 1
                    n_kps += 1
                keypoints.append(lkp)
                keypoints.append(rkp)
                for im in whole_body_imgs:
                    if im['id'] == an['image_id']:
                        image_size = [im['height'],im['width']]
                        img_file_name = im['file_name']
                an['keypoints'] = sum(keypoints,[])
                an['num_keypoints'] = n_kps

                new_an = {}
                new_an['image_file_name'] = img_file_name
                new_an['image_size'] = image_size
                for k,v in an.items():
                    if k in req_kps:
                        new_an[k] = an[k]
                new_annotations.append(new_an)
                break



  0%|          | 0/149813 [00:00<?, ?it/s]

In [7]:
with open('rebatrain_keypoints.json' , 'w') as json_file:
    json.dump(new_annotations,json_file)